# Загрузим и обработаем данные

In [4]:
import pandas as pd

df = pd.read_csv('healthcare-dataset-stroke-data.csv')
# Выведем голову
print(df.head())
print(df.info())
print(df.describe(include='all').T)

      id  gender   age  hypertension  heart_disease ever_married  \
0   9046    Male  67.0             0              1          Yes   
1  51676  Female  61.0             0              0          Yes   
2  31112    Male  80.0             0              1          Yes   
3  60182  Female  49.0             0              0          Yes   
4   1665  Female  79.0             1              0          Yes   

       work_type Residence_type  avg_glucose_level   bmi   smoking_status  \
0        Private          Urban             228.69  36.6  formerly smoked   
1  Self-employed          Rural             202.21   NaN     never smoked   
2        Private          Rural             105.92  32.5     never smoked   
3        Private          Urban             171.23  34.4           smokes   
4  Self-employed          Rural             174.12  24.0     never smoked   

   stroke  
0       1  
1       1  
2       1  
3       1  
4       1  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5110 e

# Обработаем категориальные данные

In [5]:
object_column = df.select_dtypes(include='object').columns
for column in object_column:
    print(column)
    print(df[column].unique())

gender
['Male' 'Female' 'Other']
ever_married
['Yes' 'No']
work_type
['Private' 'Self-employed' 'Govt_job' 'children' 'Never_worked']
Residence_type
['Urban' 'Rural']
smoking_status
['formerly smoked' 'never smoked' 'smokes' 'Unknown']


Дропаем Patient name
Разбиваем на категории по диапозонам Blood Pressure
Тоже самое с холестерином


In [7]:
import pandas as pd
import json
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

# Исходный датафрейм
df_ml = df.drop(columns=['id'], axis=1)

# Инициализируем словарь логирования
label_mappings = {}

# Удаляем пустые колонки (если вдруг остались)
df_ml = df_ml.drop(columns='', errors='ignore')

# Кодируем категориальные признаки
encoder = LabelEncoder()
categ_columns = df_ml.select_dtypes(include='object').columns

for column in categ_columns:
    encoder.fit(df_ml[column])
    df_ml[column] = encoder.transform(df_ml[column])

    # Сохраняем соответствие: оригинал → код
    mapping = {k: int(v) for k, v in zip(encoder.classes_, encoder.transform(encoder.classes_))}
    label_mappings[column] = mapping

# Сохраняем подготовленный DataFrame
df_ml.to_csv('df_ml.csv', index=False)

# Сохраняем отображения в JSON
with open('label_mappings.json', 'w', encoding='utf-8') as f:
    json.dump(label_mappings, f, indent=4, ensure_ascii=False)

print("Готово: DataFrame и словарь кодировок сохранены.")

Готово: DataFrame и словарь кодировок сохранены.


# Напишем модель

In [8]:
import pandas as pd
import optuna
from sklearn.model_selection import train_test_split


# Загрузка и подготовка данных
df = pd.read_csv('df_ml.csv')
X = df.drop(columns=['stroke'])
y = df['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42, stratify=y
)

# Категориальные признаки
cat_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']

for column in cat_features:
    df[column] = df[column].astype('category')

import lightgbm as lgb
from lightgbm import early_stopping, log_evaluation
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

def objective(trial):
    params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
    'max_depth': trial.suggest_int('max_depth', 5, 10),  # ← тут исправил
    'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 40),
    'num_leaves': trial.suggest_int('num_leaves', 64, 256),
    'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
    'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
    'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
    'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 5.0),
    'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 5.0),
    'force_col_wise': True,
    'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 10.0),
    'seed': 42
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[val_idx]

        # Создание LightGBM-датасетов
        dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_features)
        dvalid = lgb.Dataset(X_test, label=y_test, categorical_feature=cat_features)

        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            valid_sets=[dvalid],
            callbacks=[
                early_stopping(stopping_rounds=300),
                log_evaluation(period=0)  # отключить логи
            ]
        )

        y_pred = model.predict(X_test, num_iteration=model.best_iteration)
        y_pred_labels = (y_pred >= 0.5).astype(int)

        score = f1_score(y_test, y_pred_labels, average='macro')
        scores.append(score)

        trial.report(score, step=len(scores)-1)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return float(np.mean(scores))

# Optuna с прунингом
pruner = optuna.pruners.PercentilePruner(percentile=25, n_startup_trials=10, n_warmup_steps=2)
sampler=optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize', pruner=pruner, sampler=sampler)
study.optimize(objective, n_trials=100)

print("Best Params:", study.best_params)
print("Best F1 Score (macro):", study.best_value)

[I 2025-06-24 22:31:33,871] A new study created in memory with name: no-name-8547959e-c810-4a58-acd2-e9c963fcc94a


Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[20]	valid_0's binary_logloss: 0.179515
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[19]	valid_0's binary_logloss: 0.178939
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[20]	valid_0's binary_logloss: 0.176807
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[20]	valid_0's binary_logloss: 0.175137
Training until validation scores don't improve for 300 rounds


[I 2025-06-24 22:31:37,314] Trial 0 finished with value: 0.48751376937831037 and parameters: {'learning_rate': 0.01184431975182039, 'max_depth': 10, 'min_data_in_leaf': 32, 'num_leaves': 179, 'feature_fraction': 0.6624074561769746, 'bagging_fraction': 0.662397808134481, 'bagging_freq': 1, 'lambda_l1': 4.330880728874676, 'lambda_l2': 3.005575058716044, 'scale_pos_weight': 7.372653200164409}. Best is trial 0 with value: 0.48751376937831037.


Early stopping, best iteration is:
[19]	valid_0's binary_logloss: 0.180718
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[115]	valid_0's binary_logloss: 0.171495
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[121]	valid_0's binary_logloss: 0.171722
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[110]	valid_0's binary_logloss: 0.169489
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[108]	valid_0's binary_logloss: 0.168459
Training until validation scores don't improve for 300 rounds


[I 2025-06-24 22:31:41,192] Trial 1 finished with value: 0.48751376937831037 and parameters: {'learning_rate': 0.005242693862597309, 'max_depth': 10, 'min_data_in_leaf': 35, 'num_leaves': 104, 'feature_fraction': 0.6727299868828402, 'bagging_fraction': 0.6733618039413735, 'bagging_freq': 4, 'lambda_l1': 2.6237821581611893, 'lambda_l2': 2.1597250932105787, 'scale_pos_weight': 3.6210622617823773}. Best is trial 0 with value: 0.48751376937831037.


...

Early stopping, best iteration is:
[32]	valid_0's binary_logloss: 0.166622
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[111]	valid_0's binary_logloss: 0.15784
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[156]	valid_0's binary_logloss: 0.153036
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[81]	valid_0's binary_logloss: 0.158507
Training until validation scores don't improve for 300 rounds


[I 2025-06-24 22:37:25,630] Trial 98 pruned. 


Early stopping, best iteration is:
[141]	valid_0's binary_logloss: 0.152009
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[43]	valid_0's binary_logloss: 0.162786
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[64]	valid_0's binary_logloss: 0.163144
Training until validation scores don't improve for 300 rounds
Early stopping, best iteration is:
[36]	valid_0's binary_logloss: 0.164902
Training until validation scores don't improve for 300 rounds


[I 2025-06-24 22:37:28,998] Trial 99 pruned. 


Early stopping, best iteration is:
[81]	valid_0's binary_logloss: 0.157496
Best Params: {'learning_rate': 0.04561800114183699, 'max_depth': 7, 'min_data_in_leaf': 32, 'num_leaves': 237, 'feature_fraction': 0.8143814583132323, 'bagging_fraction': 0.773690303407539, 'bagging_freq': 2, 'lambda_l1': 0.1368646893345964, 'lambda_l2': 0.21439992299004984, 'scale_pos_weight': 1.8736933998697305}
Best F1 Score (macro): 0.5322650915915285


# Обучим модель


In [30]:
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix, precision_recall_curve


# 📥 Чтение данных
df = pd.read_csv('df_ml.csv')

# 🧹 Обработка пропусков
# — BMI: медиана
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# — avg_glucose_level: KNNImputer
knn = KNNImputer(n_neighbors=5)
df[['avg_glucose_level']] = knn.fit_transform(df[['avg_glucose_level']])

# — Итеративная импутация для остальных числовых
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
imp = IterativeImputer(random_state=42, max_iter=10)
df[num_cols] = imp.fit_transform(df[num_cols])


# 🏷 Категории
cat_features = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
for col in cat_features:
    df[col] = df[col].astype('category')

# 📊 train/test, SMOTE
X = df.drop(columns=['stroke'])
y = df['stroke']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

X_res, y_res = SMOTE(random_state=42).fit_resample(X_train, y_train)

# 🎯 Параметры LightGBM
best_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'learning_rate': 0.04561800114183699,
    'max_depth': 7,
    'min_data_in_leaf': 32,
    'num_leaves': 237,
    'feature_fraction': 0.8143814583132323,
    'bagging_fraction': 0.773690303407539,
    'bagging_freq': 2,
    'lambda_l1': 0.1368646893345964,
    'lambda_l2': 0.21439992299004984,
    'scale_pos_weight': 1.8736933998697305,
    'force_col_wise': True,
    'seed': 42
}

# 🧠 Обучение модели
dtrain = lgb.Dataset(X_res, label=y_res, categorical_feature=cat_features)
dvalid = lgb.Dataset(X_test, label=y_test, categorical_feature=cat_features)
model = lgb.train(
    best_params,
    dtrain,
    num_boost_round=5000,
    valid_sets=[dvalid],
    callbacks=[lgb.early_stopping(stopping_rounds=1000)]
)

probs = model.predict(X_test)
prec, rec, thr = precision_recall_curve(y_test, probs)
f1_scores = 2 * prec * rec / (prec + rec)
best = thr[f1_scores.argmax()]
print("Best threshold:", best)
y_pred = (probs >= best).astype(int)
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

model.save_model('stroke_lgbm_model.txt')

Training until validation scores don't improve for 1000 rounds
Early stopping, best iteration is:
[146]	valid_0's binary_logloss: 0.175722
Best threshold: 0.19336654603165407
Macro F1: 0.6037246663657303
              precision    recall  f1-score   support

         0.0       0.97      0.89      0.93       972
         1.0       0.19      0.52      0.28        50

    accuracy                           0.87      1022
   macro avg       0.58      0.70      0.60      1022
weighted avg       0.93      0.87      0.90      1022

[[862 110]
 [ 24  26]]
